Importation des bibliothèque

In [10]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt 
from biopandas.pdb import PandasPdb
import nglview as nv
from scipy import stats

In [2]:
def view_structure(file_pdb):
    # 1. Charger le fichier généré
    view = nv.show_file(file_pdb)
    # 2. Vider les représentations par défaut
    view.clear_representations()
    # 3. Afficher toute la molécule de manière fine
    view.add_representation('licorice', selection='all', color='element')
    # 4. Mettre en surbrillance spéciale les atomes de Phosphore (les "noeuds" des liaisons phosphodiester)
    view.add_representation('spacefill', selection='_P', color='orange', radius=0.8)
    # 5. Rajouter un "tube" qui suit et met en évidence de façon continue le squelette phosphodiester (backbone)
    view.add_representation('tube', selection='backbone', color='red', radius=0.2)
    
    # # 6. Ajout d'un dégradé de couleur pour voir l'orientation 5' -> 3'
    # # Le schéma 'resindex' colore du bleu (début/5') au rouge (fin/3')
    # view.add_representation('cartoon', selection='nucleic', color='resindex')
    
    # 7. Centrer la vue
    view.center()
    # Afficher l'interface
    return view

In [7]:
view_structure("final_arn.pdb")

NGLWidget()

In [ ]:
ppdb_df =  PandasPdb().read_pdb("fichier_arn/initial_optimized.pdb")
atom_df = ppdb_df.df["ATOM"]
atom_df.head(20)

In [ ]:
from Bio.PDB import Structure, Model, Chain, Residue, Atom, PDBIO

def generer_structure_depart_c3(seq, distance_moy=5.8, output="initial_c3.pdb"):
    # Création de la hiérarchie
    structure = Structure.Structure("RNA")
    model = Model.Model(0)
    chain = Chain.Chain("A")
    
    for i, nt in enumerate(seq):
        # Création du résidu (nucléotide)
        res = Residue.Residue((" ", i+1, " "), nt, " ")
        # Création de l'atome C3' à sa position x
        atom = Atom.Atom("C3'", [i * distance_moy, 0, 0], 0, 0, " ", "C3'", i+1, "C")
        res.add(atom)
        chain.add(res)
        
    model.add(chain)
    structure.add(model)
    
    # Sauvegarde
    io = PDBIO()
    io.set_structure(structure)
    io.save(output)


def voir_configuration_c3(fichier_pdb):
    """
    Visualisation NGL adaptée au modèle C3'-seulement.
    """
    view = nv.show_file(fichier_pdb)
    view.clear_representations()
    view.add_representation("ball+stick", selection="all", color="cyan", radius=0.25)
    view.center()
    return view


# Exemple rapide :
generer_structure_depart_c3(seq="ACGU"*30, distance_moy=5.8)
voir_configuration_c3("fichier_arn/initial_c3_beads.pdb")

In [ ]:
def test_shapiro(df_path):
    df = pd.read_csv(df_path)
    stat, p = stats.shapiro(df['Distance'])
    print(f"Statistique de Shapiro-Wilk : {stat:.3f}")
    print(f"p-value : {p}")
    alpha = 0.05
    if p > alpha:
        print("On ne peut pas rejeter l'hypothèse nulle (H0). Les données semblent suivre une loi normale.")
    else:
        print("On rejette l'hypothèse nulle (H0). Les données ne suivent probablement pas une loi normale.")

test_shapiro("distances_1-5.csv")

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats

# Création du graphique Q-Q
df = pd.read_csv("distances_1-5.csv")
data = df["Distance"]
stats.probplot(data, dist="norm", plot=plt)
plt.title("Graphique Q-Q pour vérifier la normalité")
plt.show()

In [1]:
from classe.BeadSpringsRASPOptimizer import BeadSpringsRASPOptimizer

model = BeadSpringsRASPOptimizer("fichier_arn/initial_c3_beads.pdb")
model.run_optimization()


Utilisation du device : cuda
🚀 Début de l'optimisation (Adam, 5 cycles de 100 epochs)...

--- Cycle 1/5 ---
Epoch   0 | Total: -239.47 | RASP: -239.47 | FENE: -0.00
Epoch  99 | Total: -29926.17 | RASP: -165.03 | FENE: -29761.15
Secousse aléatoire ! (Bruit translation: 1.50Å, rotation: 0.50rad)

--- Cycle 2/5 ---
Epoch   0 | Total: -6628.17 | RASP: -169.84 | FENE: -6458.33
Epoch  99 | Total: -26553.67 | RASP: -171.03 | FENE: -26382.64
Secousse aléatoire ! (Bruit translation: 1.12Å, rotation: 0.38rad)

--- Cycle 3/5 ---
Epoch   0 | Total: -14280.30 | RASP: -157.71 | FENE: -14122.59
Epoch  99 | Total: -30060.35 | RASP: -161.67 | FENE: -29898.68
Secousse aléatoire ! (Bruit translation: 0.75Å, rotation: 0.25rad)

--- Cycle 4/5 ---
Epoch   0 | Total: -14766.62 | RASP: -159.76 | FENE: -14606.87
Epoch  99 | Total: -29839.55 | RASP: -166.64 | FENE: -29672.90
Secousse aléatoire ! (Bruit translation: 0.38Å, rotation: 0.12rad)

--- Cycle 5/5 ---
Epoch   0 | Total: -21413.10 | RASP: -164.80 | FENE:

/home/erwann/.conda/envs/Stage/lib/python3.10/site-packages/biopandas/pdb/pandas_pdb.py:732: UserWarning: Column rasp_type is not an expected column and will be skipped.
  warn(


In [4]:
def voir_configuration_c3(fichier_pdb):
    """
    Visualisation NGL adaptée au modèle C3'-seulement.
    """
    view = nv.show_file(fichier_pdb)
    view.clear_representations()
    view.add_representation("ball+stick", selection="all", color="cyan", radius=0.25)
    view.center()
    return view

In [9]:
voir_configuration_c3("output_bead_optimized.pdb")

NGLWidget()

In [11]:
import MDAnalysis as mda
import numpy as np
import subprocess
import os

def reconstruire_avec_mda(sequence, fichier_c3_input, fichier_final="arn_final.pdb"):
    # --- 1. Générer le template Full-Atom (ton code habituel) ---
    template_pdb = "template_droit.pdb"
    script_tleap = "gen_template.in"
    
    with open(script_tleap, "w") as f:
        f.write("source leaprc.RNA.OL3\n")
        formatted_seq = " ".join([n.upper() for n in sequence])
        f.write(f"mol = sequence {{ {formatted_seq} }}\n")
        f.write(f"savepdb mol {template_pdb}\n")
        f.write("quit\n")
    
    subprocess.run(["tleap", "-f", script_tleap], check=True, stdout=subprocess.PIPE)

    # --- 2. Charger les deux structures dans MDAnalysis ---
    # u_target : contient uniquement tes 24 C3'
    # u_template : l'ARN complet généré par tleap
    u_target = mda.Universe(fichier_c3_input)
    u_template = mda.Universe(template_pdb)
    
    target_c3 = u_target.select_atoms("name C3'")
    template_residuos = u_template.residues

    print(f"Reconstruction de {len(template_residuos)} résidus...")

    # --- 3. Translation résidu par résidu ---
    for i, res in enumerate(template_residuos):
        # Trouver l'atome C3' correspondant dans le template
        c3_template = res.atoms.select_atoms("name C3'")[0]
        
        # Coordonnées cible (X, Y, Z) depuis ton fichier initial
        pos_cible = target_c3[i].position
        pos_actuelle = c3_template.position
        
        # Calculer le vecteur de translation
        vecteur_translation = pos_cible - pos_actuelle
        
        # Déplacer l'ensemble du résidu (Sucre + Base + Phosphate)
        res.atoms.positions += vecteur_translation

    # --- 4. Sauvegarder le résultat ---
    u_template.atoms.write(fichier_final)
    print(f"✅ Structure reconstruite sauvegardée dans : {fichier_final}")

    # Nettoyage
    for f in [template_pdb, script_tleap, "leap.log"]:
        if os.path.exists(f): os.remove(f)

# --- EXECUTION ---
# Ta séquence de 24 nucléotides
ma_seq = "CCUGGUAUUGCAUGUACCUCCAGG" 

# Supposons que ton fichier de C3' s'appelle 'trace_c3.pdb'
reconstruire_avec_mda(ma_seq, "output_bead_optimized.pdb")

Reconstruction de 24 résidus...
✅ Structure reconstruite sauvegardée dans : arn_final.pdb


/home/erwann/.conda/envs/Stage/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/erwann/.conda/envs/Stage/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:777: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn("Unit cell dimensions not found. "
/home/erwann/.conda/envs/Stage/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'elements' Using default value of ' '
  warnings.warn("Found no information for attr: '{}'"
/home/erwann/.conda/envs/Stage/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'formalcharges' Using default value 

In [ ]:
std15 = df15["Distance"].median()
print(f"Mediane pour 1-5 Angstrom : {std15:.3f}")

Mediane pour 1-5 Angstrom : 5.789
Mediane pour 0-1 Angstrom : 5.707


In [27]:
df15 = pd.read_csv("distances_individuelles_1-5.csv")
df15["Distance_A"].describe()

count    650896.000000
mean          5.845299
std           0.430980
min           3.217800
25%           5.656100
50%           5.796200
75%           6.014000
max          59.998300
Name: Distance_A, dtype: float64

In [5]:
df01["Distance"].describe()

count    732.000000
mean       5.709660
std        0.280333
min        4.335461
25%        5.578009
50%        5.707477
75%        5.825384
max        6.838767
Name: Distance, dtype: float64

In [8]:
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

# Chemin vers votre fichier .cif
fichier_cif = "dataset/0-1/1j8g.cif"

# Création du dictionnaire
data_dict = MMCIF2Dict(fichier_cif)

# Exemple d'accès aux données
# Les clés suivent la nomenclature officielle mmCIF (ex: _entity.pdbx_description)
# Extraction de la séquence (souvent une liste avec une chaîne par entité)
sequences = data_dict["_entity_poly.pdbx_seq_one_letter_code"]

for i, seq in enumerate(sequences):
    # Nettoyage des retours à la ligne
    clean_seq = seq.replace("\n", "")
    print(f"Entité {i+1} - Longueur : {len(clean_seq)}")
    print(f"Séquence : {clean_seq}") # Affiche les 50 premiers caractères

# Accès aux colonnes spécifiques
atom_names = data_dict["_atom_site.label_atom_id"]
x_coords = data_dict["_atom_site.Cartn_x"]
y_coords = data_dict["_atom_site.Cartn_y"]
z_coords = data_dict["_atom_site.Cartn_z"]

# Nombre total d'atomes
nb_atomes = len(atom_names)
print(f"Nombre total d'atomes : {nb_atomes}")

# Exemple : Afficher les coordonnées du premier atome
# Rappel : les valeurs sont des chaînes, on les convertit en float
for i in range(nb_atomes):
    print(f"Atome {atom_names[i]} : x={float(x_coords[i])}, y={float(y_coords[i])}, z={float(z_coords[i])}")

Entité 1 - Longueur : 6
Séquence : UGGGGU
Nombre total d'atomes : 674
Atome O5' : x=12.967, y=20.885, z=19.962
Atome C5' : x=11.871, y=21.775, z=20.189
Atome C4' : x=10.687, y=21.092, z=20.826
Atome O4' : x=10.075, y=20.168, z=19.906
Atome C3' : x=10.972, y=20.204, z=22.048
Atome O3' : x=11.163, y=21.019, z=23.2
Atome C2' : x=9.73, y=19.303, z=22.076
Atome O2' : x=8.586, y=19.995, z=22.55
Atome C1' : x=9.525, y=19.058, z=20.569
Atome N1 : x=10.211, y=17.832, z=20.092
Atome C2 : x=9.45, y=16.688, z=19.96
Atome O2 : x=8.293, y=16.632, z=20.347
Atome N3 : x=10.083, y=15.613, z=19.386
Atome C4 : x=11.408, y=15.541, z=18.999
Atome O4 : x=11.84, y=14.494, z=18.539
Atome C5 : x=12.147, y=16.766, z=19.162
Atome C6 : x=11.525, y=17.852, z=19.684
Atome P : x=11.99, y=20.451, z=24.456
Atome OP1 : x=12.471, y=21.63, z=25.215
Atome OP2 : x=12.967, y=19.418, z=24.006
Atome O5' : x=10.877, y=19.676, z=25.291
Atome C5' : x=10.24, y=20.262, z=26.434
Atome C4' : x=9.394, y=21.463, z=26.1
Atome O4' : x=8

In [13]:
atom_types = data_dict["_atom_site.label_atom_id"]
indices_c3p = [i for i, atom in enumerate(atom_types) if atom == "C3'"]

# 2. Extraire les coordonnées correspondantes et convertir en float
coords_c3p = np.array([
    [float(data_dict["_atom_site.Cartn_x"][i]),
     float(data_dict["_atom_site.Cartn_y"][i]),
     float(data_dict["_atom_site.Cartn_z"][i])]
    for i in indices_c3p
])

print(f"Nombre de résidus (C3') trouvés : {len(coords_c3p)}")
data_dict["_atom_site.label_asym_id"]

Nombre de résidus (C3') trouvés : 24


['A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B',
 'B'

In [ ]:
import numpy as np
import os
import csv
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

# Dossier source
input_folder = "dataset/0-1/"
output_file = "resultats_distances_c3p.csv"
ref_atom = "P"

# Liste pour stocker toutes les lignes du futur CSV
all_rows = []

for filename in os.listdir(input_folder):
    if not filename.endswith(".cif"):
        continue
        
    nom_fichier = filename.split(".")[0]
    
    try:
        # 1. Charger le dictionnaire
        data_dict = MMCIF2Dict(os.path.join(input_folder, filename))

        # 2. Extraire les colonnes nécessaires
        atom_names = data_dict["_atom_site.label_atom_id"]
        alt_ids = data_dict["_atom_site.label_alt_id"]
        asym_ids = data_dict["_atom_site.label_asym_id"]
        model_nums = data_dict["_atom_site.pdbx_PDB_model_num"]
        x_coords = data_dict["_atom_site.Cartn_x"]
        y_coords = data_dict["_atom_site.Cartn_y"]
        z_coords = data_dict["_atom_site.Cartn_z"]

        # 3. Identifier toutes les chaînes uniques
        chaines_uniques = sorted(list(set(asym_ids)))

        for chaine in chaines_uniques:
            # Filtrage
            indices = [
                i for i, name in enumerate(atom_names)
                if name == ref_atom
                and asym_ids[i] == chaine 
                and model_nums[i] == "1" 
                and alt_ids[i] in [".", "A"]
            ]
            
            if len(indices) >= 2:
                coords = np.array([
                    [float(x_coords[i]), float(y_coords[i]), float(z_coords[i])]
                    for i in indices
                ])
                
                # Calcul des distances
                diffs = np.diff(coords, axis=0)
                distances = np.linalg.norm(diffs, axis=1)
                moyenne = np.mean(distances)
                
                # Ajout d'une ligne pour cette chaîne dans ce fichier
                all_rows.append([nom_fichier, chaine, len(indices), round(moyenne, 4)])
                
    except Exception as e:
        print(f"Erreur sur le fichier {filename} : {e}")

# 4. Écriture du fichier CSV final
with open(output_file, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    # En-tête
    writer.writerow(["Fichier", "Chaine", "Nombre_"+ref_atom, "Distance_Moyenne"])
    # Données
    writer.writerows(all_rows)

print(f"\nTerminé ! Les résultats sont enregistrés dans : {output_file}")

Traitement de : 3dw6
Traitement de : 4rbq
Traitement de : 6zyb
Traitement de : 7qsh
Traitement de : 5nqi
Traitement de : 5ebi
Traitement de : 6kj4
Traitement de : 4rne
Traitement de : 7k9c
Traitement de : 5v2h
Traitement de : 5da6
Traitement de : 4y27
Traitement de : 8vjt
Traitement de : 6kj2
Traitement de : 6kj3
Traitement de : 7qua
Traitement de : 4nlf
Traitement de : 1q9a
Traitement de : 4rkv
Traitement de : 7ogw
Traitement de : 3nj6
Traitement de : 4xk0
Traitement de : 4pno
Traitement de : 7zug
Traitement de : 4nmg
Traitement de : 6y0y
Traitement de : 4jrd
Traitement de : 4rj1
Traitement de : 2wna
Traitement de : 5i40
Traitement de : 6j60
Traitement de : 5d99
Traitement de : 5w0g
Traitement de : 5mrx
Traitement de : 1j8g
Traitement de : 5zgl
Traitement de : 3dw5
Traitement de : 7qp2
Traitement de : 5k8h
Traitement de : 3dw4
Traitement de : 8pfq
Traitement de : 9en6
Traitement de : 7l3r
Traitement de : 6euw
Traitement de : 6kj1
Traitement de : 3s6e
Traitement de : 3dvz
Traitement de